# DNA–DNA Residue Contact Frequency：交接版

## 1. 這份 Notebook 在做什麼？

本程式使用 MDTraj 與 `contact-map`，計算分子動力學軌跡中 **DNA residue 彼此形成接觸的 frame 比例**，並輸出 DNA–DNA residue contact-frequency heatmap。

當任一對指定原子的距離小於 `CONTACT_CUTOFF_NM`，該 residue pair 在該 frame 會被視為接觸。Contact frequency 範圍為 0–1：

- `0`：所有取樣 frame 都沒有接觸
- `0.5`：一半取樣 frame 有接觸
- `1`：所有取樣 frame 都有接觸

### 執行後會得到

| 輸出檔案 | 內容 |
|---|---|
| `dna_contact_frequency.csv` | 含 residue 標籤的 contact-frequency matrix |
| `dna_contact_frequency.npz` | 矩陣、labels、參數與 residue index，供重新繪圖 |
| `dna_contact_frequency.png` | 300 dpi heatmap |
| `dna_contact_frequency.pkl` | `contact-map` 原生物件，可用 `ContactFrequency.from_file()` 重載 |
| `run_summary.txt` | 本次分析使用的檔案與參數 |

## 2. 執行順序

1. 安裝套件。
2. **只修改「使用者設定區」**。
3. 依序執行輸入檢查、計算、儲存與繪圖。
4. 檢查 DNA residue 數量、labels、cutoff 與取樣間隔。

> 第一次接手時，建議先用短 DCD 測試。確認 labels 與 heatmap 正確後，再分析完整軌跡。


## 3. 環境準備

建議建立獨立 Conda 環境：

```bash
conda create -n dna-contact python=3.11 -y
conda activate dna-contact
pip install mdtraj contact-map numpy matplotlib jupyter
jupyter notebook
```

本 Notebook 使用：

- MDTraj：讀取 PDB／DCD 與拓撲資訊
- contact-map：計算 atom contact 並彙整為 residue contact frequency
- NumPy：建立、保存 contact matrix
- Matplotlib：繪製 heatmap


In [ ]:
# ============================================================
# 4. 載入套件
# ============================================================

from collections import Counter
from pathlib import Path
import csv

import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
from contact_map import ContactFrequency
from matplotlib.colors import LinearSegmentedColormap

print(f"MDTraj version: {md.__version__}")
print(f"NumPy version: {np.__version__}")


## 5. 使用者設定區

一般情況只需修改下一格。

- `PDB_FILE`：DCD 對應的拓撲 PDB。
- `DCD_FILES`：可放一個或多個 DCD，必須依模擬時間順序排列。
- `STRIDE = 100`：每 100 個原始 frame 取一個 frame。
- `TIME_PER_FRAME_NS = 0.002`：NAMD timestep 2 fs、每 1000 steps 輸出一個 DCD frame。
- `CONTACT_CUTOFF_NM = 1.2`：保留原始程式設定，相當於 12 Å。
- `N_NEIGHBORS_IGNORED = 2`：忽略同一 chain 中前後兩個相鄰 residues，保留 `contact-map` 的原始預設行為。
- `HEAVY_ATOMS_ONLY = True`：排除氫原子，降低氫位置與建模差異造成的影響。

> `CONTACT_CUTOFF_NM` 與 `N_NEIGHBORS_IGNORED` 會直接改變 contact frequency，修改前應先確認研究定義，並在 Methods 中留下紀錄。


In [ ]:
# ============================================================
# 6. 使用者設定區：一般情況只修改這一格
# ============================================================

# ---------- 必須修改：輸入檔案 ----------
PDB_FILE = Path("/path/to/structure.pdb")

DCD_FILES = [
    Path("/path/to/trajectory.dcd"),
]

# ---------- 必須確認：分析參數 ----------
STRIDE = 100
TIME_PER_FRAME_NS = 0.002
CONTACT_CUTOFF_NM = 1.2
N_NEIGHBORS_IGNORED = 2
HEAVY_ATOMS_ONLY = True

# 常見 DNA residue names：
# CHARMM：ADE, THY, CYT, GUA
# AMBER／PDB：DA, DT, DC, DG 或 A, T, C, G
DNA_RESIDUE_NAMES = {
    "ADE", "THY", "CYT", "GUA",
    "DA", "DT", "DC", "DG",
    "A", "T", "C", "G",
}

# ---------- 輸出設定 ----------
OUTPUT_DIR = Path("./contact_results")
OUTPUT_PREFIX = "dna_contact_frequency"


## 7. 輸入檢查與軌跡載入

下一格會檢查：

- PDB 與所有 DCD 是否存在
- stride、cutoff 與時間設定是否合理
- 是否辨識到 DNA residues
- DNA atom selection 是否為空
- residue labels 是否與 PDB 編號一致

多個 DCD 會依 `DCD_FILES` 的順序串接，並在載入時套用 `STRIDE`。


In [ ]:
# ============================================================
# 8. 檢查輸入並載入軌跡
# ============================================================

def require_existing_file(path, label):
    """確認輸入檔存在，並回傳解析後的絕對路徑。"""
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            f"找不到{label}：{path}\n"
            "請回到『使用者設定區』檢查路徑。"
        )
    return path.resolve()


def is_heavy_atom(atom):
    """依 MDTraj element 或 atom name 判斷是否為非氫原子。"""
    if atom.element is not None:
        return atom.element.symbol.upper() != "H"
    return not atom.name.upper().startswith("H")


if not isinstance(STRIDE, int) or STRIDE < 1:
    raise ValueError("STRIDE 必須是大於或等於 1 的整數。")

if TIME_PER_FRAME_NS <= 0:
    raise ValueError("TIME_PER_FRAME_NS 必須大於 0。")

if CONTACT_CUTOFF_NM <= 0:
    raise ValueError("CONTACT_CUTOFF_NM 必須大於 0。")

if not isinstance(N_NEIGHBORS_IGNORED, int) or N_NEIGHBORS_IGNORED < 0:
    raise ValueError("N_NEIGHBORS_IGNORED 必須是大於或等於 0 的整數。")

pdb_path = require_existing_file(PDB_FILE, "PDB 拓撲檔")

if not DCD_FILES:
    raise ValueError("DCD_FILES 不可為空，請至少填入一個 DCD。")

dcd_paths = [
    require_existing_file(path, f"DCD 檔案（第 {i} 個）")
    for i, path in enumerate(DCD_FILES, start=1)
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# MDTraj 可依序載入多個 DCD；stride 在讀檔階段套用以節省記憶體。
traj = md.load(
    [str(path) for path in dcd_paths],
    top=str(pdb_path),
    stride=STRIDE,
)

if traj.n_frames == 0:
    raise ValueError("軌跡載入後沒有任何 frame，請檢查 DCD 與 STRIDE。")

topology = traj.topology

dna_residues = [
    residue
    for residue in topology.residues
    if residue.name.upper() in DNA_RESIDUE_NAMES
]

if not dna_residues:
    observed_names = sorted({res.name for res in topology.residues})
    raise ValueError(
        "找不到任何符合 DNA_RESIDUE_NAMES 的 residue。\n"
        f"拓撲中的 residue names：{observed_names}\n"
        "請回到使用者設定區補上正確名稱。"
    )

dna_atom_indices = [
    atom.index
    for residue in dna_residues
    for atom in residue.atoms
    if (not HEAVY_ATOMS_ONLY or is_heavy_atom(atom))
]

if not dna_atom_indices:
    raise ValueError("DNA atom selection 為空，請檢查 residue names 與原子資訊。")

# PDB resSeq 可能在不同 chain 重複；重複時自動加入 chain index。
base_labels = [f"{res.name}{res.resSeq}" for res in dna_residues]
duplicated_labels = {
    label for label, count in Counter(base_labels).items() if count > 1
}
dna_labels = [
    f"C{res.chain.index}:{label}" if label in duplicated_labels else label
    for res, label in zip(dna_residues, base_labels)
]
dna_residue_indices = np.asarray(
    [residue.index for residue in dna_residues],
    dtype=int,
)

sampling_interval_ns = STRIDE * TIME_PER_FRAME_NS

print("輸入檢查完成")
print(f"PDB: {pdb_path}")
print(f"DCD files: {len(dcd_paths)}")
print(f"Loaded sampled frames: {traj.n_frames}")
print(f"Sampling interval: {sampling_interval_ns:.3f} ns")
print(f"DNA residues: {len(dna_residues)}")
print(f"DNA atoms used: {len(dna_atom_indices)}")
print(f"Heavy atoms only: {HEAVY_ATOMS_ONLY}")
print(f"Residue labels: {', '.join(dna_labels)}")


## 9. Contact-frequency 計算與矩陣轉換

`contact-map` 內部使用 **MDTraj topology residue index（從 0 開始）** 記錄接觸，而不是 PDB 的 `resSeq`。

因此本 Notebook 不直接用全拓撲 residue index 畫刻度，而是：

1. 以 DNA atom indices 建立 `ContactFrequency`。
2. 讀取 residue-pair frequency。
3. 轉成只包含 DNA residues 的緊密 `N × N` 對稱矩陣。
4. 再以 PDB residue labels 繪圖。

這可避免系統中還有水、離子或 ligand 時，heatmap 刻度與 DNA residue 對錯位置。


In [ ]:
# ============================================================
# 10. 計算函數
# ============================================================

def calculate_dna_contact_matrix(
    traj,
    dna_atom_indices,
    dna_residue_indices,
    cutoff_nm,
    n_neighbors_ignored,
):
    """
    計算 DNA residue contact frequency 並建立緊密對稱矩陣。

    Parameters
    ----------
    traj : mdtraj.Trajectory
        已套用 stride 的軌跡。
    dna_atom_indices : sequence of int
        參與 contact 判定的 DNA atom indices。
    dna_residue_indices : sequence of int
        DNA residues 在完整 MDTraj topology 中的 residue indices。
    cutoff_nm : float
        任一 atom pair 被視為接觸的距離 cutoff，單位 nm。
    n_neighbors_ignored : int
        同一 chain 中要忽略的相鄰 residue 數量。

    Returns
    -------
    contact_frequency : contact_map.ContactFrequency
        保留完整 atom／residue contact 資訊的原生物件。
    matrix : numpy.ndarray
        只包含選定 DNA residues 的 N × N 對稱 frequency matrix。
    """
    contact_frequency = ContactFrequency(
        trajectory=traj,
        query=list(dna_atom_indices),
        haystack=list(dna_atom_indices),
        cutoff=cutoff_nm,
        n_neighbors_ignored=n_neighbors_ignored,
    )

    residue_to_matrix = {
        int(residue_index): matrix_index
        for matrix_index, residue_index in enumerate(dna_residue_indices)
    }
    matrix = np.zeros(
        (len(dna_residue_indices), len(dna_residue_indices)),
        dtype=float,
    )

    for residue_pair, frequency in (
        contact_frequency.residue_contacts.counter.items()
    ):
        pair = list(residue_pair)
        if len(pair) != 2:
            continue

        residue_i, residue_j = map(int, pair)
        if residue_i not in residue_to_matrix or residue_j not in residue_to_matrix:
            continue

        matrix_i = residue_to_matrix[residue_i]
        matrix_j = residue_to_matrix[residue_j]
        matrix[matrix_i, matrix_j] = frequency
        matrix[matrix_j, matrix_i] = frequency

    return contact_frequency, matrix


In [ ]:
# ============================================================
# 11. 執行計算
# ============================================================

trajectory_contacts, contact_matrix = calculate_dna_contact_matrix(
    traj=traj,
    dna_atom_indices=dna_atom_indices,
    dna_residue_indices=dna_residue_indices,
    cutoff_nm=CONTACT_CUTOFF_NM,
    n_neighbors_ignored=N_NEIGHBORS_IGNORED,
)

if contact_matrix.shape != (len(dna_labels), len(dna_labels)):
    raise RuntimeError("Contact matrix 大小與 DNA labels 數量不一致。")

if not np.allclose(contact_matrix, contact_matrix.T):
    raise RuntimeError("Contact matrix 不是對稱矩陣，請檢查轉換流程。")

nonzero_pairs = np.count_nonzero(np.triu(contact_matrix, k=1))

print("Contact-frequency calculation completed")
print(f"Matrix shape: {contact_matrix.shape}")
print(f"Non-zero residue pairs: {nonzero_pairs}")
print(
    "Frequency range: "
    f"{contact_matrix.min():.3f}–{contact_matrix.max():.3f}"
)


## 12. 儲存計算結果

計算與繪圖分離：完成計算後先保存 NPZ、CSV 與 `contact-map` 原生物件。之後若只想修改配色、字體或圖片尺寸，可從 NPZ 重新畫圖，不必再次讀取 DCD。

NPZ 會記錄：

- `contact_matrix`
- `dna_labels`
- 完整 topology 中的 `dna_residue_indices`
- cutoff、stride、時間間隔與相鄰 residue 排除設定


In [ ]:
# ============================================================
# 13. 儲存 CSV、NPZ、contact-map 物件與執行摘要
# ============================================================

csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.csv"
npz_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.npz"
object_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.pkl"
summary_path = OUTPUT_DIR / "run_summary.txt"

# CSV 第一列與第一欄皆為 residue label，方便人工核對矩陣。
with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["residue", *dna_labels])
    for label, row in zip(dna_labels, contact_matrix):
        writer.writerow([label, *[f"{value:.6f}" for value in row]])

np.savez_compressed(
    npz_path,
    contact_matrix=contact_matrix,
    dna_labels=np.asarray(dna_labels),
    dna_residue_indices=dna_residue_indices,
    pdb_file=str(pdb_path),
    dcd_files=np.asarray([str(path) for path in dcd_paths]),
    stride=STRIDE,
    time_per_frame_ns=TIME_PER_FRAME_NS,
    sampling_interval_ns=sampling_interval_ns,
    cutoff_nm=CONTACT_CUTOFF_NM,
    n_neighbors_ignored=N_NEIGHBORS_IGNORED,
    heavy_atoms_only=HEAVY_ATOMS_ONLY,
)

# contact-map 官方格式，可用 ContactFrequency.from_file() 重新載入。
trajectory_contacts.save_to_file(str(object_path))

summary_text = (
    "DNA–DNA residue contact-frequency analysis\n"
    f"PDB: {pdb_path}\n"
    f"DCD files: {len(dcd_paths)}\n"
    + "\n".join(f"  - {path}" for path in dcd_paths)
    + "\n"
    f"Loaded sampled frames: {traj.n_frames}\n"
    f"Stride: {STRIDE}\n"
    f"Time per original frame: {TIME_PER_FRAME_NS} ns\n"
    f"Sampling interval: {sampling_interval_ns} ns\n"
    f"Cutoff: {CONTACT_CUTOFF_NM} nm "
    f"({CONTACT_CUTOFF_NM * 10:.2f} Å)\n"
    f"Neighbors ignored: {N_NEIGHBORS_IGNORED}\n"
    f"Heavy atoms only: {HEAVY_ATOMS_ONLY}\n"
    f"DNA residues: {len(dna_residues)}\n"
    f"DNA atoms used: {len(dna_atom_indices)}\n"
    f"Non-zero residue pairs: {nonzero_pairs}\n"
)
summary_path.write_text(summary_text, encoding="utf-8")

print(f"Saved CSV: {csv_path.resolve()}")
print(f"Saved NPZ: {npz_path.resolve()}")
print(f"Saved contact-map object: {object_path.resolve()}")
print(f"Saved summary: {summary_path.resolve()}")


## 14. 從 NPZ 讀取並繪圖

下一格刻意重新讀取 NPZ，以確認保存結果完整，並示範日後如何不重跑 DCD 就修改圖片。

Heatmap 採白色到紅色：顏色越紅，代表該 residue pair 在軌跡中形成接觸的比例越高。


In [ ]:
# ============================================================
# 15. 載入 NPZ 並繪製 contact-frequency heatmap
# ============================================================

plot_data = np.load(npz_path)
plot_matrix = plot_data["contact_matrix"]
plot_labels = plot_data["dna_labels"].astype(str)

red_cmap = LinearSegmentedColormap.from_list(
    "WhiteToRed",
    ["white", "red"],
)

fig, ax = plt.subplots(figsize=(16, 14))
image = ax.imshow(
    plot_matrix,
    cmap=red_cmap,
    vmin=0,
    vmax=1,
    origin="lower",
    interpolation="nearest",
    aspect="equal",
)

positions = np.arange(len(plot_labels))
ax.set_xticks(positions)
ax.set_yticks(positions)
ax.set_xticklabels(plot_labels, rotation=90)
ax.set_yticklabels(plot_labels)

ax.set_xlabel("DNA Residue", fontsize=30)
ax.set_ylabel("DNA Residue", fontsize=30)
ax.set_title("DNA Residue Contact Frequency", fontsize=30, pad=16)
ax.tick_params(axis="both", which="major", labelsize=16)

# Minor ticks 放在每個方格邊界，用來畫網格。
boundaries = np.arange(-0.5, len(plot_labels), 1)
ax.set_xticks(boundaries, minor=True)
ax.set_yticks(boundaries, minor=True)
ax.grid(which="minor", color="gray", linestyle="--", linewidth=0.5)
ax.tick_params(which="minor", bottom=False, left=False)

colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label("Contact Frequency", fontsize=24)
colorbar.ax.tick_params(labelsize=18)

fig.tight_layout()

figure_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure: {figure_path.resolve()}")


## 16. 結果檢查與常見問題

### 跑完後至少確認五件事

1. `DNA residues` 數量是否符合研究系統。
2. `Residue labels` 是否與 PDB 中的殘基編號一致。
3. `Sampling interval` 是否正確；目前 `100 × 0.002 = 0.2 ns`。
4. cutoff 是否確實要使用 `1.2 nm = 12 Å`。
5. 是否確實要忽略同一 chain 中前後 2 個 residues。

### 常見錯誤

**找不到 PDB 或 DCD**  
回到使用者設定區修正路徑。Linux 路徑區分大小寫。

**找不到 DNA residues**  
查看錯誤訊息列出的 residue names，將正確名稱加入 `DNA_RESIDUE_NAMES`。

**記憶體不足**  
增大 `STRIDE` 或先使用較短的 DCD 測試。注意：增大 stride 會降低時間取樣密度。

**heatmap 幾乎全部是紅色**  
cutoff 可能過大。原程式的 `1.2 nm` 等於 `12 Å`；若研究定義是一般重原子接觸，常見設定會更短，但不可在不同系統間任意更換。

**相鄰 residues 都沒有訊號**  
`N_NEIGHBORS_IGNORED = 2` 會忽略同一 chain 中前後兩個 residue。若研究目的需要相鄰鹼基接觸，必須經確認後改為 `0`。

**想重畫圖片**  
從「載入 NPZ 並繪圖」開始執行即可，不需重跑 contact-frequency 計算。

### 交接時不可省略的研究定義

- Contact 的定義是任一指定 atom pair 距離小於 cutoff。
- `contact-map` 的 cutoff 單位為 nm；圖中的值是 frequency，不是距離。
- 是否只使用 heavy atoms，以及是否忽略相鄰 residues，皆會改變結果。
- `TIME_PER_FRAME_NS = 0.002` 只適用於 timestep 2 fs 且每 1000 steps 輸出 DCD 的軌跡。
